# Llama-3.2-1B, INT4 linear

Обычная модель против той же модели, где все `nn.Linear` заменены на `QuantizedLinear`.
WikiText-2 test, окно 2048, stride 512. Forward: batch 1, длины 128, 512, 2048.

Считается `backend="reference"`: веса распаковываются и умножаются в PyTorch.
Последняя ячейка на CUDA сравнивает это с Triton-кернелем.

Colab, GPU, ветка `feature/llama-int4-eval`. Первая ячейка клонирует репозиторий.
`QUICK = True` — короткий прогон. Для всего теста поставь `False`.
На T4 будет fp16, bf16 включается с compute capability 8.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feature/llama-int4-eval"
REPO_URL = "https://github.com/AITH-Triton-Quantization-Project/triton_quantization.git"
COLAB_ROOT = Path("/content/triton_quantization")

if IN_COLAB:
    get_ipython().system("pip install -q transformers datasets accelerate")
    if not (COLAB_ROOT / "int4_quantization").exists():
        get_ipython().system(f"git clone -b {BRANCH} {REPO_URL} {COLAB_ROOT}")

ROOT = Path.cwd().resolve()
candidates = (ROOT, ROOT.parent, COLAB_ROOT)
ROOT = next((path for path in candidates if (path / "int4_quantization").exists()), ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not (ROOT / "int4_quantization").exists():
    raise SystemExit(
        "Не найден пакет int4_quantization. "
        f"Сначала запушь ветку {BRANCH}."
    )

print("repo:", ROOT)


In [ ]:
import gc

import torch

from int4_quantization.llama_eval import (
    LLAMA_1B_LINEAR_LAYERS,
    MODEL_ID,
    benchmark_forward,
    check_triton_against_reference,
    generate_reply,
    load_causal_lm,
    load_wikitext2,
    perplexity,
    replace_linear_layers,
    select_device_and_dtype,
    weight_footprint,
)

# False — весь WikiText-2 и длины 128/512/2048
QUICK = True

MAX_LENGTH = 2048
STRIDE = 512
MAX_TOKENS = 2048 if QUICK else None
SPEED_SEQ_LENS = (128,) if QUICK else (128, 512, 2048)
WARMUP = 1 if QUICK else 2
ITERS = 1 if QUICK else 5
PROMPT = "What is the capital of France? Answer in one short sentence."
MAX_NEW_TOKENS = 32


## Устройство


In [ ]:
device, dtype = select_device_and_dtype()
print("device:", device)
print("dtype:", dtype)
if device.type == "cpu":
    print("GPU не виден. В Colab включите среду выполнения с GPU.")


## Обычная модель

После замера веса удаляются, чтобы не держать вторую копию на GPU.


In [ ]:
def megabytes(num_bytes):
    return num_bytes / 1024**2


def show_footprint(title, stats):
    print(title)
    print(f"  linear: {stats['linear_layers']}, quantized: {stats['quantized_layers']}")
    print(f"  веса: {megabytes(stats['linear_bytes']):.1f} МБ")
    print(f"  коды int4: {megabytes(stats['code_bytes']):.1f} МБ")
    print(f"  коды + scale/zero: {megabytes(stats['quantized_bytes']):.1f} МБ")


tokenizer, baseline = load_causal_lm(MODEL_ID, dtype, device)
vocab_size = baseline.get_input_embeddings().num_embeddings
show_footprint("fp16/bf16", weight_footprint(baseline))
baseline_reply = generate_reply(baseline, tokenizer, PROMPT, device, MAX_NEW_TOKENS)
print(baseline_reply)


In [ ]:
baseline_speed = benchmark_forward(
    baseline,
    seq_lens=SPEED_SEQ_LENS,
    vocab_size=vocab_size,
    device=device,
    warmup=WARMUP,
    iters=ITERS,
)
baseline_speed


In [ ]:
input_ids = load_wikitext2(tokenizer, max_tokens=MAX_TOKENS)
print("tokens:", int(input_ids.shape[1]))
baseline_ppl = perplexity(
    baseline,
    input_ids,
    max_length=MAX_LENGTH,
    stride=STRIDE,
    log_every=10,
)
baseline_ppl


In [ ]:
del baseline
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()


## INT4

Меняются все `nn.Linear`, включая `lm_head`. Embedding остаётся. Для 1B это 113 слоёв.


In [ ]:
tokenizer, quantized = load_causal_lm(MODEL_ID, dtype, device)
replaced = replace_linear_layers(quantized, backend="reference")
print("заменено:", len(replaced))
if len(replaced) != LLAMA_1B_LINEAR_LAYERS:
    print("ожидалось", LLAMA_1B_LINEAR_LAYERS, replaced[:8], "...", replaced[-1])
show_footprint("int4", weight_footprint(quantized))
quantized_reply = generate_reply(quantized, tokenizer, PROMPT, device, MAX_NEW_TOKENS)
print(quantized_reply)


In [ ]:
quantized_speed = benchmark_forward(
    quantized,
    seq_lens=SPEED_SEQ_LENS,
    vocab_size=vocab_size,
    device=device,
    warmup=WARMUP,
    iters=ITERS,
)
quantized_ppl = perplexity(
    quantized,
    input_ids,
    max_length=MAX_LENGTH,
    stride=STRIDE,
    log_every=10,
)
quantized_speed, quantized_ppl


## Triton

Одно умножение кернелем против эталона. `ok` — можно переключать слой на `backend="triton"`.


In [ ]:
triton_check = check_triton_against_reference(device)
triton_check


## Итог

Время int4 здесь включает распаковку весов в PyTorch. К скорости кернеля это относится только при статусе `ok` выше.


In [ ]:
def format_speed(rows):
    return {
        int(row["seq_len"]): f"{row['ms']:.1f} мс, {row['tokens_per_s']:.1f} ток/с"
        for row in rows
    }


print("quick:" if QUICK else "full:", device, dtype)
print(f"ppl bf16/fp16: {baseline_ppl['perplexity']:.3f}")
print(f"ppl int4:      {quantized_ppl['perplexity']:.3f}")
print("speed bf16/fp16:", format_speed(baseline_speed))
print("speed int4:     ", format_speed(quantized_speed))
print("triton:", triton_check["status"], triton_check.get("message", ""))
